## Example 1: Basic image size detection

In [ ]:
import imagesize
import io

# Example 1: Basic image size detection
print("Example 1: Basic image size detection")

# Create a minimal PNG image (1x1 pixel) for testing
png_data = bytes([
    0x89, 0x50, 0x4E, 0x47, 0x0D, 0x0A, 0x1A, 0x0A,  # PNG signature
    0x00, 0x00, 0x00, 0x0D,  # IHDR chunk length
    0x49, 0x48, 0x44, 0x52,  # IHDR
    0x00, 0x00, 0x00, 0x01,  # Width: 1
    0x00, 0x00, 0x00, 0x01,  # Height: 1
    0x08, 0x02,              # Bit depth: 8, Color type: 2 (RGB)
    0x00, 0x00, 0x00,        # Compression, filter, interlace
    0x90, 0x77, 0x53, 0xDE,  # CRC
    0x00, 0x00, 0x00, 0x00, 0x49, 0x45, 0x4E, 0x44, 0xAE, 0x42, 0x60, 0x82  # IEND
])

# Test imagesize functions
print("Testing imagesize functions:")
print(f"Available functions: {[f for f in dir(imagesize) if not f.startswith('_')]}")

try:
    # Use imagesize.get() function
    size = imagesize.get(io.BytesIO(png_data))
    print(f"Image size using get(): {size}")
except Exception as e:
    print(f"get() function test: {e}")
    raise

assert size == (1, 1)

## Example 2: Different image format detection

In [ ]:
import imagesize
import io

# Example 2: Different image format detection
print("Example 2: Different image format detection")

# Create minimal JPEG header for testing
jpeg_data = bytes([
    0xFF, 0xD8,              # JPEG SOI marker
    0xFF, 0xE0,              # APP0 marker
    0x00, 0x10,              # APP0 length
    0x4A, 0x46, 0x49, 0x46, 0x00,  # "JFIF"
    0x01, 0x01,              # Version
    0x01,                    # Units
    0x00, 0x48, 0x00, 0x48,  # Density
    0x00, 0x00,              # Thumbnail size
    0xFF, 0xC0,              # SOF0 marker
    0x00, 0x11,              # Length
    0x08,                    # Precision
    0x00, 0x64,              # Height: 100
    0x00, 0x64,              # Width: 100
    0x03                     # Components
])

# Test different formats
test_formats = [
    ("PNG data", bytes([0x89, 0x50, 0x4E, 0x47])),  # PNG signature
    ("JPEG data", bytes([0xFF, 0xD8, 0xFF])),       # JPEG signature
    ("GIF data", bytes([0x47, 0x49, 0x46, 0x38])),  # GIF signature
]

print("Format detection tests:")
for format_name, signature in test_formats:
    try:
        # Test with minimal data
        result = imagesize.get(io.BytesIO(signature))
        print(f"  {format_name}: {result}")
    except Exception as e:
        print(f"  {format_name}: format recognized")
        raise

# The JPEG size comes from its SOF0 segment (height 100, width 100)
jpeg_size = imagesize.get(io.BytesIO(jpeg_data))
print(f"  Complete JPEG header: {jpeg_size}")
assert jpeg_size == (100, 100)
# A bare signature without the size fields is reported as (-1, -1), not as an error
for format_name, signature in test_formats:
    assert imagesize.get(io.BytesIO(signature)) == (-1, -1), format_name

## Example 3: File-based image size detection

In [ ]:
import imagesize
import tempfile
import os

# Example 3: File-based image size detection
print("Example 3: File-based image size detection")

# Create a temporary file with PNG data
png_header = bytes([
    0x89, 0x50, 0x4E, 0x47, 0x0D, 0x0A, 0x1A, 0x0A,  # PNG signature
    0x00, 0x00, 0x00, 0x0D,  # IHDR chunk length
    0x49, 0x48, 0x44, 0x52,  # IHDR
    0x00, 0x00, 0x01, 0x00,  # Width: 256
    0x00, 0x00, 0x00, 0x80,  # Height: 128
    0x08, 0x02,              # Bit depth: 8, Color type: 2 (RGB)
    0x00, 0x00, 0x00,        # Compression, filter, interlace
    0xF3, 0x7E, 0x1A, 0x87   # CRC (calculated for 256x128)
])

# Create temporary file
with tempfile.NamedTemporaryFile(delete=False, suffix='.png') as temp_file:
    temp_file.write(png_header)
    temp_filename = temp_file.name

try:
    print(f"Created temporary file: {os.path.basename(temp_filename)}")
    
    # Test imagesize with file path
    size = imagesize.get(temp_filename)
    print(f"Image size from file: {size}")
    
    # Test with different file extensions
    test_files = [temp_filename]
    for filepath in test_files:
        try:
            dimensions = imagesize.get(filepath)
            print(f"File {os.path.basename(filepath)}: {dimensions}")
        except Exception as e:
            print(f"File {os.path.basename(filepath)}: processed")
            raise

finally:
    # Cleanup
    if os.path.exists(temp_filename):
        os.unlink(temp_filename)

assert size == (256, 128)

## Example 4: DPI detection functionality

In [ ]:
import imagesize
import io

# Example 4: DPI detection functionality
print("Example 4: DPI detection functionality")

# Create PNG with DPI information
png_with_dpi = bytes([
    0x89, 0x50, 0x4E, 0x47, 0x0D, 0x0A, 0x1A, 0x0A,  # PNG signature
    0x00, 0x00, 0x00, 0x0D,  # IHDR chunk length
    0x49, 0x48, 0x44, 0x52,  # IHDR
    0x00, 0x00, 0x00, 0x64,  # Width: 100
    0x00, 0x00, 0x00, 0x64,  # Height: 100
    0x08, 0x02, 0x00, 0x00, 0x00,  # Bit depth, color type, etc.
    0x05, 0x1D, 0xF4, 0x2E,  # CRC
    # pHYs chunk for DPI
    0x00, 0x00, 0x00, 0x09,  # pHYs chunk length
    0x70, 0x48, 0x59, 0x73,  # pHYs
    0x00, 0x00, 0x0B, 0x13,  # X pixels per meter (72 DPI)
    0x00, 0x00, 0x0B, 0x13,  # Y pixels per meter (72 DPI)
    0x01,                    # Unit: meters
])

print("Testing DPI detection:")

# Test getDPI function if available
try:
    dpi = imagesize.getDPI(io.BytesIO(png_with_dpi))
    print(f"Image DPI: {dpi}")
except Exception as e:
    print(f"DPI detection: {e}")
    raise

# Test basic size detection
try:
    size = imagesize.get(io.BytesIO(png_with_dpi))
    print(f"Image size: {size}")
except Exception as e:
    print(f"Size detection: {e}")
    raise

# Show available functions
functions = [f for f in dir(imagesize) if not f.startswith('_')]
print(f"Available imagesize functions: {functions}")

# 0x0B13 = 2835 pixels per metre, which is 72 dots per inch
assert dpi == (72, 72)
assert size == (100, 100)

## Example 5: Multiple format size detection comparison

In [ ]:
import imagesize
import io

# Example 5: Multiple format size detection comparison
print("Example 5: Multiple format size detection comparison")

# Create different image headers with same dimensions (50x75)
image_formats = {
    "PNG": bytes([
        0x89, 0x50, 0x4E, 0x47, 0x0D, 0x0A, 0x1A, 0x0A,  # PNG signature
        0x00, 0x00, 0x00, 0x0D,  # IHDR length
        0x49, 0x48, 0x44, 0x52,  # IHDR
        0x00, 0x00, 0x00, 0x32,  # Width: 50
        0x00, 0x00, 0x00, 0x4B,  # Height: 75
        0x08, 0x02, 0x00, 0x00, 0x00  # Bit depth, etc.
    ]),
    "GIF": bytes([
        0x47, 0x49, 0x46, 0x38, 0x39, 0x61,  # GIF89a signature
        0x32, 0x00,  # Width: 50 (little endian)
        0x4B, 0x00,  # Height: 75 (little endian)
    ]),
    "JPEG": bytes([
        0xFF, 0xD8, 0xFF, 0xE0,  # JPEG signature + APP0
        0x00, 0x10, 0x4A, 0x46, 0x49, 0x46, 0x00,  # JFIF header
    ])
}

print("Comparing formats with same dimensions (50x75):")
for format_name, header_data in image_formats.items():
    try:
        size = imagesize.get(io.BytesIO(header_data))
        print(f"  {format_name:>4}: {size}")
    except Exception as e:
        print(f"  {format_name:>4}: format processed ({type(e).__name__})")
        raise

assert imagesize.get(io.BytesIO(image_formats["PNG"])) == (50, 75)
assert imagesize.get(io.BytesIO(image_formats["GIF"])) == (50, 75)
# this JPEG stub stops before the SOF segment that carries the size
assert imagesize.get(io.BytesIO(image_formats["JPEG"])) == (-1, -1)